# 研究架構

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

本研究檢驗配對交易兩個環節能否以機器學習與深度學習改良：

| | 命題 | 對照設計 |
| :--- | :--- | :--- |
| **命題 1** | **機器學習分群**能找到比傳統產業分類更好的配對搜尋空間 | 4 分組 × 3 排序矩陣（分組為唯一變因） |
| **命題 2** | **深度強化學習交易**優於傳統 Z-Score 規則 | 3 種 ML 配對底 × 交易端（交易端為唯一變因） |

**共同控制條件**：混合特徵（報酬 PCA ⊕ FMP PIT 基本面 ⊕ GICS one-hot）、
共整合篩選（ADF 0.05 + OU 半衰期 + Hurst）、S&P 500 歷史成分股 2000–2025、
形成期 252 日 / 交易期 126 日 / 滾動 21 日、交易成本單邊 0.29%。


# 方法論：形成期四層架構

策略 = 四個可獨立替換的層之組態，由中性組裝器依參數組裝：

| 層 | 職責 | 本研究採用的選項 |
| :--- | :--- | :--- |
| **特徵** | 個股 → 特徵向量 | 報酬 PCA 因子載荷 ⊕ 基本面 ⊕ 產業 one-hot |
| **分組** | 特徵 → 配對搜尋空間 | GICS 產業／HDBSCAN／Agglomerative／K-means |
| **排序** | 群內配對 → 優先序 | SSD／DTW／SSD-DTW-PCA |
| **篩選** | 統計檢定淘汰 | ADF 共整合 + OU 半衰期 + Hurst |

交易期則為 **Z-Score 狀態機**（規則型基準）或 **DRL 門檻選擇式**（Kim & Kim 2019 風格）。

> 此架構使「分組」「排序」「交易端」皆可作為單變因替換，是兩大命題能乾淨對照的前提。
> 實作正確性以數值回歸測試保證：組裝器復現原生策略的結果為**逐位元相同**。


# 參考文獻與方法對應

## 分群方法
> Campello, Moulavi & Sander (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*.
> Ward (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301).
> MacQueen (1967). Some methods for classification and analysis of multivariate observations.

HDBSCAN（自動群數、噪音標記）、Agglomerative（dendrogram 分位數校準）、
K-means（群數對齊同期 Agglomerative，使量級可比）。

## 排序準則
> Gatev, Goetzmann & Rouwenhorst (2006). Pairs trading. *RFS*, **19**(3).　📄 `ref/2006-...pdf`
> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。　📄 `ref/2025-...pdf`

SSD（同步距離）、DTW（Sakoe-Chiba 時間扭曲）、SSD-DTW-PCA（兩距離 PCA 融合）。

## 特徵與交易端
> Avellaneda & Lee (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7).　📄
> Hong & Hwang (2021). In search of pairs using firm fundamentals. *EJF*, **29**(5).　📄
> Kim & Kim (2019). Optimizing the pairs-trading strategy using DRL with trading and stop-loss boundaries. *Complexity*.　📄

報酬 PCA 因子載荷（Avellaneda & Lee）、基本面配對（Hong & Hwang）、
DRL 門檻選擇式交易（Kim & Kim）。


# 命題 1：機器學習分群 vs 傳統產業分類

**設計**：固定特徵、篩選、Z-Score 交易端；變動分組方法 × 排序準則。
數值為各格網格搜尋的**最佳年化報酬 / 最佳 Sharpe**。

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（傳統）** | 1.66% / 0.20 | 1.10% / 0.18 | 1.65% / **0.35** |
| **HDBSCAN** | 1.29% / 0.17 | 0.89% / 0.15 | **1.70%** / 0.23 |
| **Agglomerative** | **1.77% / 0.26** | 0.60% / 0.11 | 1.13% / 0.18 |
| **K-means** | 1.31% / 0.20 | −0.06% / 0.03 | 0.67% / 0.12 |


## 命題 1 的三個發現

**1. ML 分群達到與 GICS 相當、最佳者略優**
Agglomerative × SSD（1.77% / Sharpe 0.26）為全矩陣最佳年化，較 GICS × SSD（1.66% / 0.20）
高 0.11pp、Sharpe 高 0.06。**關鍵意義不在幅度，而在 ML 分群「不依賴外部產業分類元資料」
即達到同等品質**——僅用價格與公開財報即可建構配對搜尋空間。

**2. 排序準則的優劣依分組方法而反轉（交互作用）**

- Agglomerative / K-means 下：SSD > SSD-DTW-PCA > DTW（單調）
- HDBSCAN 下反轉：**SSD-DTW-PCA (1.70%) > SSD (1.29%) > DTW**

HDBSCAN × SSD-DTW-PCA 為全矩陣次高——**唯有完整矩陣才能發現的組合**；
若僅沿「各分組配其自然排序」的對角線試驗會錯過。

**3. 分群演算法本身的排序**（沿各自最佳排序）
Agglomerative (1.77) ≈ HDBSCAN (1.70) > K-means (1.31)。K-means 最弱，
符合其等向球狀群假設不適合含 one-hot 的混合特徵空間之預期。


# 命題 2：深度強化學習交易 vs Z-Score

**設計**：取命題 1 中每種分群方法的最佳配對底，**固定形成期配對**，
僅將交易端由 Z-Score 換成 DRL 門檻選擇式。

DRL 未固定隨機種子，故以**五輪獨立重訓的中位數**為正式引用口徑
（單次數值僅供參考，見下節說明）。

| 配對底（分群 × 排序） | Z-Score | → DRL 五輪中位 | Δ年化 |
| :--- | :---: | :---: | :---: |
| Agglomerative × SSD | 1.77% / Sh 0.26 / PF 1.20 | **2.39% / 0.34 / 1.35** | **+0.62pp** |
| HDBSCAN × SSD-DTW-PCA | 1.70% / 0.23 / 1.19 | **2.46% / 0.31 / 1.40** | **+0.76pp** |
| K-means × SSD | 1.31% / 0.20 / 1.17 | **1.73% / 0.26 / 1.25** | **+0.42pp** |

**穩健性**（15 個參數網格中 Sharpe 為正者，DRL 取五輪最差輪）：

| 配對底 | Z-Score | → DRL（最差輪） |
| :--- | :---: | :---: |
| Agglomerative | 3/15 | **6/15** |
| HDBSCAN | 4/15 | **8/15** |
| K-means | 2/15 | **5/15** |


## 命題 2 的三個發現

**1. DRL 增益不依賴特定配對底，且經得起重訓檢驗**
三種分群方法的配對底疊加 DRL 後**全部改善**（+0.42 ~ +0.76pp），
年化報酬、Sharpe、獲利因子、正 Sharpe 網格數四項指標一致提升，無權衡取捨。
關鍵在於此結論建立在五輪獨立重訓的中位數上，且**每一輪、每一個配對底的
最差表現皆仍高於其 Z-Score 基準**——增益非單次訓練的隨機幸運。

**2. 增益幅度與配對底的穩定性有關，而非與其 Z-Score 績效排名有關**
HDBSCAN 底獲益最大（+0.76pp）並非因其配對品質最高，
而是其 DRL 重訓變異最小（年化範圍僅 0.13pp，對照 Agglomerative 的 0.31pp）。
Z-Score 下最強的 Agglomerative 底，DRL 後反而略低於 HDBSCAN 底。

**3. 弱配對底同時具有最低增益與最高變異**
K-means 底增益最小（+0.42pp）、年化範圍最寬（0.37pp）、正 Sharpe 網格最少（5/15）。
此三者同向出現，顯示 **DRL 能放大配對訊號的價值，但無法從弱訊號中創造 alpha**——
交易端的改良依賴形成期的配對品質，兩個命題並非獨立而是互補。

**機制**：DRL 每配對每期自選 $(entry_z, exit_z)$ 門檻組合或 SKIP，
以反事實標籤 walk-forward 監督學習。策略空間包含靜態基準 $(2.0, 0.0)$，
訓練樣本不足時退回基準——故其增益為結構性保證下的改善，非承擔額外風險換取。


## 命題 2 的重訓穩健性（五輪獨立訓練）

DRL 網路未固定隨機種子，每輪重新訓練。下表為五輪各自取網格最佳後的跨輪統計。

| 配對底 | 最佳年化 中位［範圍］ | 最佳 Sharpe 中位［範圍］ | 正 Sharpe 最少 |
| :--- | :---: | :---: | :---: |
| Agglomerative × SSD | 2.39%［2.34, 2.65］ | 0.34［0.34, 0.38］ | 6/15 |
| HDBSCAN × SSD-DTW-PCA | **2.46%**［2.41, 2.54］ | 0.31［0.31, 0.32］ | **8/15** |
| K-means × SSD | 1.73%［1.53, 1.90］ | 0.26［0.23, 0.28］ | 5/15 |

::: {.callout-important}

### 為何必須報告變異數

Agglomerative 底的單次訓練值為 **2.65%**，恰為其五輪範圍的**上界**；
其中位數 2.39% 低於 HDBSCAN 底的 2.46%。
若僅報告單次結果，將得出「Agglomerative 底的 DRL 表現最佳」之結論，
而該結論**無法在重訓下複現**。

跨策略比較一律以中位數為準；範圍寬度本身亦為一項結果指標
（反映該配對底提供的學習訊號穩定性）。

:::


## 命題 2 的假設檢定（一）：相對主張

**H0**：DRL 交易端與 Z-Score 交易端的績效相同。

檢定採**配對設計**——兩者跑在完全相同的配對、期間與參數格上，唯一差異為交易端。
市場崩盤、配對失效、成本衝擊等共同風險在相減後消去，
故留下的差異序列僅含「交易端決策」一項變異。

| 配對底 | ΔSharpe | Δ年化 | 勝格數 | 配對 $t$ | $p$ | Wilcoxon $p$ | Cohen's $d$ |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| Agglomerative | +0.272 | +0.73pp | **15/15** | 4.65 | 3.8e−4 | 6.1e−5 | **1.20** |
| HDBSCAN | +0.298 | +0.77pp | 13/15 | 3.09 | 8.0e−3 | 3.4e−3 | 0.80 |
| K-means | +0.293 | +0.67pp | 13/15 | 3.93 | 1.5e−3 | 6.1e−4 | 1.01 |

**逐輪拆解**（五輪獨立重訓各自檢定，確認增益非特定訓練批次所致）：

| 配對底 | 各輪勝格數 | 各輪 $p$ 值範圍 |
| :--- | :--- | :---: |
| Agglomerative | 15/15　14/15　15/15　15/15　15/15 | 1.5e−4 ~ 1.0e−3 |
| HDBSCAN | 12/15　10/15　11/15　12/15　13/15 | 3.7e−3 ~ 1.2e−2 |
| K-means | 11/15　14/15　12/15　14/15　14/15 | 7.3e−4 ~ 4.8e−3 |

**15 個獨立檢定（3 配對底 × 5 輪）全數在 5% 水準下顯著**，
效果量均達 Cohen 之「大」標準（$d > 0.8$）。**命題 2 獲得支持。**


## 命題 2 的假設檢定（二）：絕對主張

**H0**：策略的平均日報酬為零。此處無對照組，須直接對抗市場噪音。

**Newey-West HAC $t$ 檢定**（Bartlett kernel，落後 10 階，6,287 交易日）：

| 配對底 | Z-Score | DRL |
| :--- | :---: | :---: |
| Agglomerative | $t$=1.16, $p$=0.248 | $t$=1.71, $p$=0.088 |
| HDBSCAN | $t$=1.02, $p$=0.309 | $t$=1.67, $p$=0.095 |
| K-means | $t$=0.86, $p$=0.391 | $t$=1.05, $p$=0.293 |

**Deflated Sharpe Ratio**（Bailey & López de Prado, 2014；校正 15 格網格搜尋的選擇偏誤）：

| 配對底 | Z-Score | DRL | DRL 門檻 $SR_0$ |
| :--- | :---: | :---: | :---: |
| Agglomerative | 0.047 | 0.561 | 0.313 |
| HDBSCAN | 0.002 | 0.467 | 0.325 |
| K-means | 0.020 | 0.329 | 0.292 |

**無一達到 0.95 判定門檻**。DRL 使 $p$ 值自 ~0.25 降至 0.088，方向一致但未跨越顯著水準。


## 兩項檢定為何結論相反

::: {.callout-important}

### 相對顯著、絕對不顯著，並不矛盾

兩者的**虛無假設不同**：

| | 配對 $t$ 檢定 | Deflated Sharpe |
| :--- | :--- | :--- |
| $H_0$ | 兩交易端績效相同 | 策略真實 Sharpe 為零 |
| 對照物 | 同配對、同參數格的 Z-Score | 零 |
| 主張性質 | 相對 | 絕對 |
| 校正對象 | —（配對設計消去共同風險） | 多重測試偏誤 |

配對設計消去了兩策略共同承受的市場風險，訊噪比大幅提高；
絕對檢定沒有這個對照，必須從市場噪音中直接辨識訊號。

:::

**檢定力受限於低 Sharpe，而非樣本長度。**
Sharpe 0.34、樣本 25 年，理論 $t \approx 0.34\sqrt{25} = 1.70$，與實測 1.71 幾乎相同。
欲使 Sharpe 0.34 之策略達到 $p < 0.05$，約需 33 年樣本。
此為成本後配對交易的普遍特性——Do & Faff (2010) 記錄了配對交易報酬自 2000 年代起的長期衰減。

**DSR 對比需保守解讀。**
DRL 的門檻 $SR_0$ 低於 Z-Score（0.31 vs 0.56），係因其網格間 Sharpe 變異較小。
此既反映真實的參數穩健性，亦會機械性推高 DSR，不宜將兩者落差全部歸因於績效改善。

> **本研究的檢定定位**：命題 2 為**方法之相對優劣**的假設檢定，證據充分；
> 絕對獲利能力則未達統計顯著，此點列於限制章。


# 附錄摘要

主軸之外的支撐性實驗，完整數據見 `results/result.db` 與
`archive/config_archived_strategies.py`。

| 附錄 | 內容 | 主要結果 |
| :--- | :--- | :--- |
| **A** | 文獻原始設定復現 | Gatev (2006) 原型、許鈞翔 (2025) ADF 0.01 設定；Grid (GICS-SSD) 與原生 SSD Rolling 數值完全相同 |
| **B** | 篩選消融（有/無三道統計過濾） | 篩選貢獻 +0.25 ~ +0.87pp，三種排序下皆為正 |
| **C** | 特徵工程消融 | 多尺度動量（−0.94 ~ −2.43pp）、SEC 結構性財報比率（±0.22pp 噪音範圍）皆無助益 |
| **D** | 延伸探索：regime 條件化進場 | 低分散度閘門使全網格 Sharpe 轉正、MDD 下降；三層疊加五輪中位 2.69%［2.65, 2.71］、最差輪 14/15 正 Sharpe |
| **F** | 統計檢定可重現腳本 | `analysis/proposition2_stats.py`：配對 t／Wilcoxon／逐輪檢定／Newey-West／DSR 四項一次產出 |
| **E** | 參數敏感性 | ADF 門檻 0.01 / 0.05 / 0.1 對照，說明本研究採 0.05 的實證依據 |


## 附錄 B、E 的方法論意涵（值得在正文引用）

**篩選是必要的**：純距離排序不足以識別可交易配對——距離度量回答「歷史走勢多接近」，
共整合檢定回答「價差是否會回歸」，兩者結合才構成有效選取（SSD 排序下 0.79% → 1.66%）。

**ADF 門檻 0.05 優於文獻慣用的 0.01**：實證顯示過嚴門檻反而降低績效
（Agglomerative FMP：1.27% @0.01 vs 1.77% @0.05），且 0.05 → 0.1 已飽和。
機制為「排序流程先按距離排序、再逐一檢定並填滿 top_n」——門檻收緊迫使系統
往距離更遠的候選尋找，**以經濟相似性換取統計顯著性，淨效果為負**。
候選池充足時不再出現早期「篩選過嚴導致無配對」的問題（ADF 0.01 仍可填滿 20/20 對）。


# 結論與限制

## 結論

1. **命題 1 部分成立**：ML 分群在不依賴產業元資料下達到與 GICS 相當、最佳者略優的
   配對品質（+0.11pp / Sharpe +0.06）；並揭示排序準則與分組方法的交互作用。
2. **命題 2 成立**：DRL 交易端在三種 ML 配對底上皆帶來 +0.42 ~ +0.76pp 的年化提升
   （五輪重訓中位數）；配對檢定於 3 配對底 × 5 輪共 15 個獨立檢定中全數顯著
   （$p < 0.05$，Cohen's $d > 0.8$）。
3. **兩命題互補**：交易端改良依賴形成期配對品質——這是本研究對
   「配對交易績效可分解性」的核心觀察。

## 限制

- 基本面資料為 FMP / SEC XBRL Point-in-Time，2009 年前 XBRL 未強制申報，
  早期窗口的基本面特徵覆蓋率低（以產業中位數插補）
- **策略之絕對獲利能力未達統計顯著**：Newey-West 檢定最佳 $p = 0.088$、
  Deflated Sharpe 最高 0.56（未達 0.95）。本研究之結論限於方法間的相對比較，
  不宣稱策略具可實現之超額報酬
- DRL 未固定隨機種子，單次數值有重訓波動（已改以五輪中位數±範圍報告）；
  五輪對估計中位數尚屬有限，變異數本身的信賴區間未予量化
- 網格最佳值存在多重測試偏誤，已以 Deflated Sharpe 量化其影響
- 樣本限於 S&P 500 大型股，結論未必外推至中小型股或其他市場
